In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./yelp_review_fine-grained_5_classes_csv/train.csv")

In [3]:
print(df.head(10))

   class_index                                        review_text
0            5  dr. goldberg offers everything i look for in a...
1            2  Unfortunately, the frustration of being Dr. Go...
2            4  Been going to Dr. Goldberg for over 10 years. ...
3            4  Got a letter in the mail last week that said D...
4            1  I don't know what Dr. Goldberg was like before...
5            5  Top notch doctor in a top notch practice. Can'...
6            5  Dr. Eric Goldberg is a fantastic doctor who ha...
7            1  I'm writing this review to give you a heads up...
8            2  Wing sauce is like water. Pretty much a lot of...
9            3  Decent range somewhat close to the city.  The ...


In [4]:
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

Total rows: 650000
Total columns: 2


In [5]:
print(df['class_index'].value_counts().sort_index())

class_index
1    130000
2    130000
3    130000
4    130000
5    130000
Name: count, dtype: int64


In [6]:
null_counts = df.isnull().sum()
print(null_counts)
#df = df.dropna(subset=['class_index, 'review_text'])

class_index    0
review_text    0
dtype: int64


In [7]:
filtered_df = df[df['class_index'].isin([1, 3, 5])]

In [8]:
balanced_df = filtered_df.groupby('class_index').sample(n=125000, random_state=42)

In [9]:
sentiment_mapping = {1: 0, 3: 1, 5: 2}
balanced_df['sentiment'] = balanced_df['class_index'].map(sentiment_mapping)

In [10]:
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True) #shuffling the dataset to spread classes evenly

In [11]:
print(balanced_df['sentiment'].value_counts().sort_index())

sentiment
0    125000
1    125000
2    125000
Name: count, dtype: int64


In [12]:
output_filename = 'yelp_balanced_300k.csv'
balanced_df[['review_text', 'sentiment']].to_csv(output_filename, index=False)

In [13]:
df = pd.read_csv("./yelp_balanced_300k.csv")

In [14]:
print(df.head(10))

                                         review_text  sentiment
0  This place is a cesspool, hands down the worst...          0
1  I feel like Distill *could* be better, but I j...          1
2  I have been going to this place since Mike ope...          2
3  Great Burgers always fresh and delicious also ...          2
4  Believe the hype - Bachi Burger is really good...          2
5  Risque would be a 3.5 star club, but the bottl...          2
6  This place finally opened up and it was defini...          2
7  Food is ok but service sucks. I though they we...          0
8  came to vegas for a vacation and heard that th...          2
9  I went to this place about once a week with my...          0


In [15]:
import re
import spacy
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

nlp = spacy.load("en_core_web_sm")

In [ ]:
def clean_and_negate_pipeline(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove basic noise (URLs and Usernames)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@[\w]+", "", text)

    # 2. Feed raw text into SpaCy
    doc = nlp(text)

    # 3. Identify tokens directly impacted by negations
    negated_tokens = set()
    negation_words = set()

    for token in doc:
        if token.dep_ == "neg":
            negation_words.add(token)  # Isolate the structural "not/n't"
            negated_tokens.add(token.head)  # Capture the target word being negated

            # Catch modifying adjectives/adverbs (e.g., "very" in "not very good")
            for child in token.head.children:
                # Protect modifiers, making sure we don't accidentally include the negation word itself
                if child.pos_ in ["ADJ", "ADV"] and child != token:
                    negated_tokens.add(child)

    # 4. Filter noise and extract Lemmatized text
    cleaned_tokens = []
    for token in doc:
        # Explicitly drop the literal negation word so it doesn't cause noise or 'not_not'
        if token in negation_words:
            continue

        # Skip raw punctuation and standalone digits
        if token.is_punct or token.is_digit:
            continue

        # Drop stop words ONLY if they aren't protected inside the negation zone
        if token.is_stop and token not in negated_tokens:
            continue

        # Clean any remaining stray symbols from the token text
        pure_word = re.sub(r"[^a-zA-Z]", "", token.text).strip()
        if not pure_word:
            continue

        # Extract SpaCy's high-accuracy dictionary root (lemma)
        lemma = token.lemma_.lower()

        # Apply our "not_" tag if the word fell inside the negation zone
        if token in negated_tokens:
            cleaned_tokens.append(f"not_{lemma}")
        else:
            cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)

In [17]:
test_cases = [
    "I didn't like the food, but I absolutely LOVED the service!", 
    "I did not like the food, the service, or the ambiance.",      
    "Hey @foodie, the 10 pizzas were disgusting, however the drinks were good.",
    "I've never experienced such a terrible place; it wasn't good.",
    "The steak was lacking flavor, despite the friendly staff."
]

for i, case in enumerate(test_cases, 1):
    print(f"Test {i} Original: {case}")
    print(f"Test {i} Cleaned:  {clean_and_negate_pipeline(case)}")
    print("-" * 50)

Test 1 Original: I didn't like the food, but I absolutely LOVED the service!
Test 1 Cleaned:  not_like food absolutely love service
--------------------------------------------------
Test 2 Original: I did not like the food, the service, or the ambiance.
Test 2 Cleaned:  not_like food service ambiance
--------------------------------------------------
Test 3 Original: Hey @foodie, the 10 pizzas were disgusting, however the drinks were good.
Test 3 Cleaned:  hey pizza disgusting drink good
--------------------------------------------------
Test 4 Original: I've never experienced such a terrible place; it wasn't good.
Test 4 Cleaned:  not_experience terrible place not_be not_good
--------------------------------------------------
Test 5 Original: The steak was lacking flavor, despite the friendly staff.
Test 5 Cleaned:  steak lack flavor despite friendly staff
--------------------------------------------------


In [18]:
df['cleaned_text'] = df['review_text'].progress_apply(clean_and_negate_pipeline)

print("Saving the finalized preprocessed dataset...")
# Save only our features and target to keep the file lightweight
df[['cleaned_text', 'sentiment']].to_csv('./yelp_preprocessed_final.csv', index=False)
print("Saved to yelp_preprocessed_final.csv")

100%|██████████| 375000/375000 [1:16:05<00:00, 82.14it/s] 


Saving the finalized preprocessed dataset...
Saved to yelp_preprocessed_final.csv


In [19]:
df = pd.read_csv('./yelp_preprocessed_final.csv')
print(df.head(10).to_string())

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                cleaned_text  sentiment
0                               

In [20]:
null_counts = df.isnull().sum()
print(null_counts)
df = df.dropna(subset=['cleaned_text', 'sentiment'])
print("\n")
null_counts = df.isnull().sum()
print(null_counts)

cleaned_text    41
sentiment        0
dtype: int64


cleaned_text    0
sentiment       0
dtype: int64


In [21]:
X = df['cleaned_text']
y = df['sentiment']

In [22]:
print(f"Full DataFrame Shape : {df.shape}")
print(f"X (Features) Type    : {type(X)} | Length: {len(X)}")
print(f"y (Target) Type      : {type(y)} | Length: {len(y)}")
print("\nClass Distribution in y:")
print(y.value_counts(normalize=True))

print("\nFirst 3 rows of X:")
print(X.head(3).to_string())

Full DataFrame Shape : (374959, 2)
X (Features) Type    : <class 'pandas.Series'> | Length: 374959
y (Target) Type      : <class 'pandas.Series'> | Length: 374959

Class Distribution in y:
sentiment
1    0.333354
2    0.333338
0    0.333308
Name: proportion, dtype: float64

First 3 rows of X:
0    place cesspool hand bad circle k work driver s...
1    feel like distill well not_just not_catch good...
2    go place mike open couple year ago warm welcom...


In [23]:
from sklearn.model_selection import train_test_split

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print Splitting Metadata
print(f"X_train Shape : {X_train.shape} | y_train Shape : {y_train.shape}")
print(f"X_test Shape  : {X_test.shape}  | y_test Shape  : {y_test.shape}")
print(f"Train ratio   : {len(X_train) / len(X) * 100:.1f}%")
print(f"Test ratio    : {len(X_test) / len(X) * 100:.1f}%")

X_train Shape : (299967,) | y_train Shape : (299967,)
X_test Shape  : (74992,)  | y_test Shape  : (74992,)
Train ratio   : 80.0%
Test ratio    : 20.0%


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- Transforming Text into TF-IDF Vectors ---")
# Using unigrams and bigrams, keeping top 75,000 most frequent tokens
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=75000, sublinear_tf=True, min_df=10)

# fit_transform learns the vocabulary AND transforms X_train
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform ONLY uses the learned vocabulary on X_test (prevents data leakage)
X_test_tfidf = vectorizer.transform(X_test)

# Print Vectorizer Metadata
print(f"Type of X_train_tfidf : {type(X_train_tfidf)}") 
print(f"X_train_tfidf Shape   : {X_train_tfidf.shape}  -> (Documents, Unique Token Features)")
print(f"X_test_tfidf Shape    : {X_test_tfidf.shape}")
print(f"Total Features Extracted : {len(vectorizer.get_feature_names_out())}")
print(f"Sparsity Check (Non-zero elements in Train) : {X_train_tfidf.nnz}")

--- Transforming Text into TF-IDF Vectors ---
Type of X_train_tfidf : <class 'scipy.sparse._csr.csr_matrix'>
X_train_tfidf Shape   : (299967, 75000)  -> (Documents, Unique Token Features)
X_test_tfidf Shape    : (74992, 75000)
Total Features Extracted : 75000
Sparsity Check (Non-zero elements in Train) : 20174476


In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Initialize Models
nb_model = MultinomialNB(alpha=1.0)
lr_model = LogisticRegression(max_iter=1000, C=1, solver='lbfgs')

# 2. Train Models
print("Training Multinomial Naive Bayes...")
nb_model.fit(X_train_tfidf, y_train)

print("Training Multinomial Logistic Regression...")
lr_model.fit(X_train_tfidf, y_train)

# 3. Predict and Evaluate
y_pred_nb = nb_model.predict(X_test_tfidf)
y_pred_lr = lr_model.predict(X_test_tfidf)

print("\n" + "="*20 + " NAIVE BAYES REPORT " + "="*20)
print(classification_report(y_test, y_pred_nb))

print("\n" + "="*20 + " LOGISTIC REGRESSION REPORT " + "="*20)
print(classification_report(y_test, y_pred_lr))

Training Multinomial Naive Bayes...
Training Multinomial Logistic Regression...

==================== NAIVE BAYES REPORT ====================
              precision    recall  f1-score   support

           0       0.84      0.83      0.84     24995
           1       0.73      0.79      0.76     24999
           2       0.88      0.81      0.85     24998

    accuracy                           0.81     74992
   macro avg       0.82      0.81      0.81     74992
weighted avg       0.82      0.81      0.81     74992


==================== LOGISTIC REGRESSION REPORT ====================
              precision    recall  f1-score   support

           0       0.89      0.90      0.90     24995
           1       0.81      0.80      0.80     24999
           2       0.88      0.88      0.88     24998

    accuracy                           0.86     74992
   macro avg       0.86      0.86      0.86     74992
weighted avg       0.86      0.86      0.86     74992



In [26]:
import joblib

# 1. Save the fitted TF-IDF Vectorizer
joblib.dump(vectorizer, './tfidf_vectorizer.joblib')
print("Saved: tfidf_vectorizer.joblib")

# 2. Save the trained Naive Bayes Model
joblib.dump(nb_model, './naive_bayes_model.joblib')
print("Saved: naive_bayes_model.joblib")

# 3. Save the trained Logistic Regression Model
joblib.dump(lr_model, './logistic_regression_model.joblib')
print("Saved: logistic_regression_model.joblib")

Saved: tfidf_vectorizer.joblib
Saved: naive_bayes_model.joblib
Saved: logistic_regression_model.joblib


In [ ]:
import joblib
import re
import spacy
nlp = spacy.load("en_core_web_sm")

loaded_vectorizer = joblib.load('./tfidf_vectorizer.joblib')
loaded_lr_model = joblib.load('./logistic_regression_model.joblib')

new_reviews = [
    "They gave me extra fries at the bottom of the bag. Legends.",
    "the food was nice but the staff were not friendly.",
    "the food was not nice and the staff were not friendly."
]

for new_review in new_reviews:

    print("\n" + "=" * 50)
    print(f"Original Review: '{new_review}'")

    cleaned_review = clean_and_negate_pipeline(new_review)
    print(f"SpaCy Preprocessed: '{cleaned_review}'")

    review_tfidf = loaded_vectorizer.transform([cleaned_review])

    prediction = loaded_lr_model.predict(review_tfidf)[0]
    probabilities = loaded_lr_model.predict_proba(review_tfidf)[0]

    print("-" * 30)
    print(f"Predicted Sentiment: {prediction}")
    print(f"Model Confidence   : {probabilities.max() * 100:.2f}%")
    print("-" * 30)


Original Review: 'Generous portion sizes! I always leave completely stuffed.'
SpaCy Preprocessed: 'generous portion size leave completely stuff'
------------------------------
Predicted Sentiment: 1
Model Confidence   : 41.27%
------------------------------

Original Review: 'the food was nice but the staff were not friendly.'
SpaCy Preprocessed: 'food nice staff not_be not_friendly'
------------------------------
Predicted Sentiment: 1
Model Confidence   : 77.45%
------------------------------

Original Review: 'the food was not nice and the staff were not friendly.'
SpaCy Preprocessed: 'food not_be not_nice staff not_be not_friendly'
------------------------------
Predicted Sentiment: 0
Model Confidence   : 70.87%
------------------------------
